# ADS — 行政区 × 工作日/周末 × 来源 指标表

**这是一张全量重算的小汇总表**，读取 Q1 2024 全量 DWS（51,240 行），join 两张维度表，
产出 55 行的 `ads/kpi_by_borough`，按 `(borough, service_zone, day_type, source_type)` 汇总。
每次运行覆盖写整张表（无 TARGET_DATE 分区），幂等。

星型模型的实际应用：
- `dws/trips_daily` → 事实表（全量读取，不按日期过滤）
- `dim_zone`（265 个区域）→ broadcast join 取 borough、service_zone
- `dim_date`（91 天）→ broadcast join 取 is_weekend

> **borough 为 `Unknown` 或 `N/A` 的记录**：`Unknown`（location_id 264）= 位置未知；
> `N/A`（location_id 265，zone "Outside of NYC"）= 上车点在纽约市边界以外，不是数据缺失。

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-ADS-Borough")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE     = os.path.expanduser("~/tlc-spark-pipeline")
DWS      = f"{BASE}/data/warehouse/dws/trips_daily"
DIM_ZONE = f"{BASE}/data/warehouse/dwd/dim_zone"
DIM_DATE = f"{BASE}/data/warehouse/dwd/dim_date"
ADS_OUT  = f"{BASE}/data/warehouse/ads/kpi_by_borough"

print(f"Spark {spark.version} ready")

## 1. 读取维度表

In [ ]:
dws      = spark.read.parquet(DWS)
dim_zone = spark.read.parquet(DIM_ZONE)
dim_date = spark.read.parquet(DIM_DATE)

dws_total = dws.agg(F.sum("trip_count")).first()[0]
print(f"DWS rows  : {dws.count():,}  |  total trip_count = {dws_total:,}")
print(f"dim_zone  : {dim_zone.count():,} zones")
print(f"dim_date  : {dim_date.count():,} days")
dim_zone.show(3)
dim_date.show(3)

## 2. Broadcast join 维度表

dim_zone（265 行）和 dim_date（91 行）均远小于 DWS，使用 `F.broadcast()` 广播到各执行器，
避免 shuffle，消除大表 join 的网络传输开销。

In [ ]:
enriched = (
    dws
    .join(
        F.broadcast(dim_zone.select("location_id", "borough", "service_zone")),
        dws["PULocationID"] == dim_zone["location_id"],
        "left"
    )
    .join(
        F.broadcast(dim_date.select("date", "is_weekend", "day_of_week")),
        dws["pickup_date"] == dim_date["date"],
        "left"
    )
    .withColumn("day_type", F.when(F.col("is_weekend"), "weekend").otherwise("weekday"))
)

# 诊断：未匹配的区域（dim_zone 覆盖全部 265 个 NYC 区域，理论上应为 0）
unmatched = enriched.filter(F.col("borough").isNull()).select("PULocationID").distinct()
unmatched_cnt = unmatched.count()
print(f"Unmatched location_ids: {unmatched_cnt}")
if unmatched_cnt > 0:
    unmatched.show()

## 3. 汇总：行政区 × 工作日/周末 × 来源

保留 6 个可加和列（sum-additive），方便上层再次聚合，同时在本表计算派生指标。

In [ ]:
kpi = (
    enriched
    .groupBy("borough", "service_zone", "day_type", "source_type")
    .agg(
        # 可加和列：供上层再次聚合
        F.sum("trip_count").alias("trip_count"),
        F.round(F.sum("gmv"),           2).alias("gmv"),
        F.round(F.sum("fare_sum"),      2).alias("fare_sum"),
        F.round(F.sum("tip_sum"),       2).alias("tip_sum"),
        F.round(F.sum("distance_sum"),  2).alias("distance_sum"),
        F.round(F.sum("duration_sum"),  2).alias("duration_sum"),
        # 派生指标：在汇总后基于加法列计算，保持数值一致性
        F.round(F.sum("fare_sum")     / F.sum("trip_count"),  2).alias("avg_fare"),
        F.round(F.sum("tip_sum")      / F.sum("fare_sum"),    4).alias("tip_rate"),
        F.round(F.sum("distance_sum") / F.sum("trip_count"),  2).alias("avg_miles"),
        F.round(F.sum("duration_sum") / F.sum("trip_count"),  1).alias("avg_min"),
    )
    .orderBy("borough", "day_type", "source_type")
)

print(f"Output rows: {kpi.count():,}")
kpi.show(20, truncate=False)

## 4. 行数守恒检查

汇总表的 `sum(trip_count)` 必须等于 DWS 输入的 `sum(trip_count)`。
如果 broadcast join 造成了行数膨胀或数据丢失，这里会报错，阻止脏数据写出。

In [ ]:
out_total = kpi.agg(F.sum("trip_count")).first()[0]
print(f"DWS  sum(trip_count) = {dws_total:,}")
print(f"KPI  sum(trip_count) = {out_total:,}")
assert out_total == dws_total, (
    f"守恒检查失败：KPI {out_total:,} != DWS {dws_total:,}，差值 {out_total - dws_total:,}"
)
print("✅ 守恒检查通过")

## 5. 写出 ADS

In [ ]:
(
    kpi.write
    .mode("overwrite")
    .parquet(ADS_OUT)
)
print(f"Written to: {ADS_OUT}")

# 快速验证：各行政区 Q1 总单量（含 Unknown/N/A）
print("\n=== Q1 各行政区总单量 ===")
(
    spark.read.parquet(ADS_OUT)
    .groupBy("borough")
    .agg(
        F.sum("trip_count").alias("trips"),
        F.round(F.sum("gmv") / 1e6, 1).alias("gmv_M")
    )
    .orderBy(F.desc("trips"))
    .show()
)

spark.stop()
print("ADS borough build complete.")